In [1]:
from __future__ import annotations

import argparse
import json
import os
import re
import unicodedata
from pathlib import Path
from typing import Any

import pytesseract
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
from PIL import ImageOps
from rapidfuzz import fuzz
from pypdf import PdfReader
from tqdm import tqdm


d:\AI-ML\AI Projects\Study RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#  importent
PROJECT_ROOT = Path.cwd().parent 

TOC_KEYWORDS = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি")
DEFAULT_MATCH_THRESHOLD = 80
POOPLER_PATH = r"C:\poppler\Library\bin"


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n\n", "\n\n", "\n", "।", " ", ""],
)



In [3]:
#  importent
with open("D:\AI-ML\AI Projects\Study RAG\chapters.json", "r", encoding="utf-8") as file:
        INDEX_INFOS = json.load(file)

<>:2: SyntaxWarning: invalid escape sequence '\A'
<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_11412\1340231910.py:2: SyntaxWarning: invalid escape sequence '\A'
  with open("D:\AI-ML\AI Projects\Study RAG\chapters.json", "r", encoding="utf-8") as file:


In [4]:

# def load_chapters(chapters_path: Path, class_no: int, subject_name: str) -> list[dict[str, Any]]:
#     with chapters_path.open("r", encoding="utf-8") as file:
#         index = json.load(file)

#     class_key = f"class_{class_no}"
#     books = index["all_books"][class_key]
#     subject_key = subject_name.casefold()
#     if subject_key == "math":
#         subject_key = "mathematics"
#     if subject_key not in books:
#         available = ", ".join(books)
#         raise ValueError(f"No index for {class_key}/{subject_name}. Available: {available}")

#     chapters = books[subject_key]
#     if not chapters:
#         raise ValueError(f"No chapters found for {class_key}/{subject_key}")
#     return sorted(chapters, key=lambda chapter: chapter["chapter_number"])


In [5]:


# def find_pdf_path(project_root: Path, class_no: int, subject_name: str) -> Path:
#     class_pdf_dir = project_root / "pdf" / f"Class_{class_no}"
#     subject_stem = "Math" if subject_name.casefold() in {"math", "mathematics"} else subject_name
#     pdf_path = next(
#         (
#             candidate
#             for candidate in class_pdf_dir.glob("*.pdf")
#             if candidate.stem.casefold() == subject_stem.casefold()
#         ),
#         class_pdf_dir / f"{subject_stem}.pdf",
#     )
#     if not pdf_path.is_file():
#         raise FileNotFoundError(f"PDF not found for class {class_no}, subject {subject_name}: {pdf_path}")
#     return pdf_path


In [6]:


# def ocr_pdf_to_pages(pdf_path: Path, poppler_path: str | None = None, lang: str = "ben", dpi: int = 300) -> list[dict[str, Any]]:
    
#     convert_options: dict[str, Any] = {"dpi": dpi}
#     if poppler_path:
#         convert_options["poppler_path"] = poppler_path

#     page_count = len(PdfReader(str(pdf_path)).pages)
#     pages = []
#     batch_size = 8
#     with tqdm(total=page_count, desc="OCR processing") as progress:
#         for first_page in range(1, page_count + 1, batch_size):
#             last_page = min(first_page + batch_size - 1, page_count)
#             images = convert_from_path(
#                 str(pdf_path),
#                 first_page=first_page,
#                 last_page=last_page,
#                 **convert_options,
#             )
#             for page_index, image in enumerate(images, start=first_page):
#                 image = ImageOps.autocontrast(image.convert("L"))
#                 text = pytesseract.image_to_string(image, lang=lang)
#                 pages.append({"pdf_page_index": page_index, "text": text})
#                 progress.update(1)
#             images.clear()
#     return pages



In [7]:
#  importent
def get_page_metadatas(class_num, subject, page):
    metas = {}
    if class_num is None or subject is None or page is None or page < 0:
        return metas

    book_section = INDEX_INFOS.get("all_books", {}).get(class_num, {}).get(subject)
    if not book_section:
        return metas

    for chapter in book_section:
        start = chapter.get("starting_page")
        if start is None or start > page:
            continue

        if chapter.get("sub_chapters"):
            for sub in chapter["sub_chapters"]:
                sub_start = sub.get("starting_page")
                if sub_start is not None and sub_start <= page:
                    metas["part"] = chapter.get("chapter_name")
                    metas["chapter"] = sub.get("sub_chapter_name")
                    if sub.get("writer_name"):
                        metas["writer_name"] = sub.get("writer_name")
                    else:
                        metas.pop("writer_name", None)
        else:
            metas["chapter"] = chapter.get("chapter_name")
            metas.pop("part", None)
            metas.pop("writer_name", None)

    return metas
    
print(get_page_metadatas("class_7", "bangla", 80))

{'part': 'কবিতা', 'chapter': 'সাম্য', 'writer_name': 'সুফিয়া কামাল'}


## self written

In [8]:
#  importent
def ocr_pdf_to_pages(pdf_path : Path, poppler_path = POOPLER_PATH, lang='ben'):

    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    keywords=("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")
    tracker = False
    page_num = 0

    page_count = len(PdfReader(str(pdf_path)).pages)
    print(page_count)
    images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)
    pages = []

    for i, img in  tqdm(enumerate(images, start=1), total=len(images), desc="Processing images"):
        img = ImageOps.autocontrast(img.convert('L'))
        text = pytesseract.image_to_string(img, lang=lang)

        if any(k in text for k in keywords):
            tracker = True

        if tracker:
            pages.append({"page_number": page_num, "page_content": text, "book_name": sub_name, "class_number": class_number})
            page_num += 1
        else:
            pages.append({"page_number": 0, "page_content": text, "book_name": sub_name, "class_number": class_number})

    return pages

In [16]:
import torch
import torchvision.transforms.functional as F
torch.cuda.is_available()

False

## By GPU Accelaretion

In [14]:
from pathlib import Path
import torch
import torchvision.transforms.functional as F
import easyocr
from pypdf import PdfReader
from pdf2image import convert_from_path
from tqdm import tqdm

# Initialize the EasyOCR reader outside the function or inside? 
# Better to initialize it once globally to avoid reloading models on every function call.
# This loads both Bengali and English models directly into GPU VRAM.
READER = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available())

def ocr_pdf_to_pages_gpu(pdf_path: Path, poppler_path=POOPLER_PATH):
    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    # Broadened keywords list to handle potential mixed characters
    keywords = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")
    tracker = False
    page_num = 0

    page_count = len(PdfReader(str(pdf_path)).pages)
    print(f"Total Pages to Process: {page_count}")
    
    # 1. Convert PDF to PIL Images (CPU-bound poppler task)
    images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)
    pages = []

    # Check if GPU is ready
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i, img in tqdm(enumerate(images, start=1), total=len(images), desc="Processing images via GPU"):
        
        # 2. Move image to GPU VRAM using PyTorch
        img_tensor = F.to_tensor(img).to(device)  # Normalizes to [0.0, 1.0], Shape: [C, H, W]
        
        # 3. GPU Preprocessing: Convert to Grayscale ('L')
        if img_tensor.shape[0] == 3:  # If RGB
            img_tensor = 0.2989 * img_tensor[0] + 0.5870 * img_tensor[1] + 0.1140 * img_tensor[2]
            img_tensor = img_tensor.unsqueeze(0)  # Shape: [1, H, W]

        # 4. GPU Preprocessing: Autocontrast (Normalize min/max scale)
        min_val, max_val = img_tensor.min(), img_tensor.max()
        if max_val > min_val:
            img_tensor = (img_tensor - min_val) / (max_val - min_val)

        # 5. Convert Tensor back to NumPy uint8 array expected by EasyOCR 
        img_numpy = (img_tensor.squeeze(0).cpu().numpy() * 255).astype('uint8')

        # 6. Run GPU-based OCR Inference
        # paragraph=True merges bounding boxes together into natural text strings
        results = READER.readtext(img_numpy, paragraph=True)
        text = "\n".join([res[1] for res in results])

        # Check for table of contents keywords
        if any(k in text for k in keywords):
            tracker = True

        if tracker:
            pages.append({"page_number": page_num, "page_content": text, "book_name": sub_name, "class_number": class_number})
            page_num += 1
        else:
            pages.append({"page_number": 0, "page_content": text, "book_name": sub_name, "class_number": class_number})

    return pages


Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

d:\AI-ML\AI Projects\Study RAG\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


In [17]:
len(PdfReader(r"D:\AI-ML\AI Projects\Study RAG\pdf\Class_6\BGS.pdf").pages)

70

In [ ]:
#  importent
book_dicts = ocr_pdf_to_pages_gpu(Path(r"D:\AI-ML\AI Projects\Study RAG\pdf\Class_6\BGS.pdf"))

documents = []
for item in tqdm(book_dicts, desc="Processing PDF pages"):
    content = item.pop("page_content", "")

    class_num = item.get("class_number")
    subject = item.get("source")
    page = item.get("page_number")

    if page is None or page < 0:
        continue  # pre-TOC / cover pages — not indexed

    metas = get_page_metadatas(class_num, subject, page)
    if not metas:
        print(f"⚠️ No metadata found for page {page}")

    doc = Document(page_content=content, metadata=item | metas)
    documents.append(doc)

print(f"{len(documents)} page-level documents")

Total Pages to Process: 70


Processing images via GPU:   0%|          | 0/70 [00:00<?, ?it/s]d:\AI-ML\AI Projects\Study RAG\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images via GPU:  36%|███▌      | 25/70 [12:00<22:50, 30.45s/it]

In [ ]:
from itertools import groupby

def group_key(doc: Document):
    m = doc.metadata
    return (m.get("class_number"), m.get("source"), m.get("part"), m.get("chapter"))

def merge_documents_by_chapter(page_documents: list[Document]) -> list[Document]:
    docs_sorted = sorted(
        page_documents,
        key=lambda d: (group_key(d), d.metadata.get("page_number", 0)),
    )

    merged = []
    for key, group in groupby(docs_sorted, key=group_key):
        group = list(group)
        merged_text = "\n".join(d.page_content for d in group)
        pages = [d.metadata.get("page_number") for d in group if d.metadata.get("page_number") is not None]

        merged_meta = dict(group[0].metadata)
        merged_meta.pop("page_number", None)
        if pages:
            merged_meta["page_start"] = min(pages)
            merged_meta["page_end"] = max(pages)

        merged.append(Document(page_content=merged_text, metadata=merged_meta))

    return merged

chapter_documents = merge_documents_by_chapter(documents)
print(f"{len(chapter_documents)} chapter-level documents")

chunked_documents = text_splitter.split_documents(chapter_documents)
print(f"{len(chunked_documents)} chunks")

[Document(metadata={'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 0, 'page_end': 65}, page_content='বাংলাদেশ ও বিশ্বপরিচয়\nষষ্ঠ শ্রেণি\n\n\nজাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড কর্তৃক ২০১২ শিক্ষাবর্ষ থেকে\nষষ্ঠ শ্রেণির পাঠ্য পুস্তকর/পে নির্ধারিত\n\nবাংলাদেশ ও বিশ্বপরিচয়\nযষ্ঠ শ্রেণি\n\n২০২৬ শিক্ষাবর্ষের জন্য পরিমার্জিত\n\n(eS ,\nay) জাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড, বাংলাদেশ\n\nজাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড, বাংলাদেশ\n৬৯-৭০, মতিঝিল বাণিজ্যিক এলাকা, ঢাকা-১০০০\nকর্তৃক প্রকাশিত'),
 Document(metadata={'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 0, 'page_end': 65}, page_content='[ প্রকাশক কর্তৃক সর্বস্বত্ব সংরক্ষিত |\n\nপ্রথম সংস্করণ রচনা ও সম্পাদনা\nঅধ্যাপক ড. মুনতাসীর মামুন\nঅধ্যাপক শফিউল আলম\nআবুল মোমেন\nঅধ্যাপক ড. মাহবুব সাদিক\nঅধ্যাপক ড. মোরশেদ শফিউল হাসান\nঅধ্যাপক ড. সৈয়দ আজিজুল হক\nসৈয়দ মাহফুজ আলী\nঅধ্যাপক মমতাজউদ্দীন পাটোয়ারী\nঅধ্যাপক ড. খোন্দকার মোকাদ্দেম হোসেন\nঅধ্যাপক ড. আবু মোঃ দেলোয়ার হোসেন\nঅধ্যাপক ড. এ কে এম শাহনাওয়াজ\nড. স